In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("yeast.csv")

X = df.drop(columns=["name"]).to_numpy(dtype=float)
y = df["name"].to_numpy()

feature_names = df.drop(columns=["name"]).columns.tolist()
classes = np.unique(y)

print("Dataset shape:", df.shape)
print("Features:", feature_names)
print("Classes:", classes)
print("\nClass distribution:")
print(df["name"].value_counts())

Dataset shape: (1484, 9)
Features: ['mcg', 'gvh', 'alm', 'mit', 'erl', 'pox', 'vac', 'nuc']
Classes: ['CYT' 'ERL' 'EXC' 'ME1' 'ME2' 'ME3' 'MIT' 'NUC' 'POX' 'VAC']

Class distribution:
name
CYT    463
NUC    429
MIT    244
ME3    163
ME2     51
ME1     44
EXC     35
VAC     30
POX     20
ERL      5
Name: count, dtype: int64


In [2]:
def gini(y):
    if len(y) == 0:
        return 0

    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)

    return 1 - np.sum(probabilities ** 2)


def entropy(y):
    if len(y) == 0:
        return 0

    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)

    return -np.sum(probabilities * np.log2(probabilities))


print("Example Gini:", gini(y))
print("Example Entropy:", entropy(y))

Example Gini: 0.7767734541306732
Example Entropy: 2.4904176362327273


In [3]:
class DecisionTreeScratch:

    def __init__(
        self,
        criterion="gini",
        max_depth=5,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight=None
    ):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.class_weight = class_weight
        self.classes_ = None
        self.root = None

    def impurity(self, y):
        if self.criterion == "gini":
            return gini(y)
        return entropy(y)

    def weighted_impurity(self, y_left, y_right):
        total = len(y_left) + len(y_right)

        return (
            len(y_left) / total * self.impurity(y_left)
            + len(y_right) / total * self.impurity(y_right)
        )

    def best_split(self, X, y):
        parent_impurity = self.impurity(y)
        best_gain = 0
        best_split = None

        for feature in range(X.shape[1]):
            values = np.unique(X[:, feature])

            if len(values) < 2:
                continue

            thresholds = (values[:-1] + values[1:]) / 2

            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask

                if (
                    left_mask.sum() < self.min_samples_leaf
                    or right_mask.sum() < self.min_samples_leaf
                ):
                    continue

                y_left = y[left_mask]
                y_right = y[right_mask]

                gain = (
                    parent_impurity
                    - self.weighted_impurity(y_left, y_right)
                )

                if gain > best_gain:
                    best_gain = gain
                    best_split = (
                        feature,
                        threshold,
                        left_mask,
                        right_mask
                    )

        return best_gain, best_split

    def majority_class(self, y):
        values, counts = np.unique(y, return_counts=True)
        return values[np.argmax(counts)]

    def build_tree(self, X, y, depth):
        prediction = self.majority_class(y)

        node = {
            "prediction": prediction,
            "counts": {
                str(c): int(np.sum(y == c))
                for c in self.classes_
            },
            "samples": len(y),
            "impurity": self.impurity(y),
            "depth": depth,
            "feature": None,
            "threshold": None,
            "left": None,
            "right": None
        }

        if (
            depth >= self.max_depth
            or len(y) < self.min_samples_split
            or len(np.unique(y)) == 1
        ):
            return node

        gain, split = self.best_split(X, y)

        if split is None or gain <= 1e-12:
            return node

        feature, threshold, left_mask, right_mask = split

        node["feature"] = feature
        node["threshold"] = threshold
        node["gain"] = gain

        node["left"] = self.build_tree(
            X[left_mask], y[left_mask], depth + 1
        )

        node["right"] = self.build_tree(
            X[right_mask], y[right_mask], depth + 1
        )

        return node

    def fit(self, X, y):
        self.classes_ = np.unique(y)

        self.root = self.build_tree(X, y, 0)

        return self

    def predict_one(self, x):
        node = self.root

        while node["feature"] is not None:
            if x[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]

        return node["prediction"]

    def predict(self, X):
        return np.array([
            self.predict_one(x) for x in X
        ])

    def predict_proba(self, X):
        probabilities = []

        for x in X:
            node = self.root

            while node["feature"] is not None:
                if x[node["feature"]] <= node["threshold"]:
                    node = node["left"]
                else:
                    node = node["right"]

            counts = np.array([
                node["counts"][str(c)]
                for c in self.classes_
            ], dtype=float)

            probabilities.append(counts / counts.sum())

        return np.array(probabilities)

    def get_depth(self, node=None):
        if node is None:
            node = self.root

        if node["feature"] is None:
            return 0

        return 1 + max(
            self.get_depth(node["left"]),
            self.get_depth(node["right"])
        )

    def get_leaves(self, node=None):
        if node is None:
            node = self.root

        if node["feature"] is None:
            return 1

        return (
            self.get_leaves(node["left"])
            + self.get_leaves(node["right"])
        )

In [4]:
def classification_metrics(y_true, y_pred, y_prob, classes):
    cm = np.zeros((len(classes), len(classes)), dtype=int)

    class_to_index = {
        c: i for i, c in enumerate(classes)
    }

    for actual, predicted in zip(y_true, y_pred):
        cm[
            class_to_index[actual],
            class_to_index[predicted]
        ] += 1

    recalls = []
    f1_scores = []

    for i in range(len(classes)):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp

        recall = tp / (tp + fn) if tp + fn else 0
        precision = tp / (tp + fp) if tp + fp else 0

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        recalls.append(recall)
        f1_scores.append(f1)

    accuracy = np.trace(cm) / cm.sum()

    # Multiclass Matthews Correlation Coefficient
    t = cm.sum(axis=1)
    p = cm.sum(axis=0)
    c = np.trace(cm)
    s = cm.sum()

    denominator = np.sqrt(
        (s**2 - np.sum(p**2))
        * (s**2 - np.sum(t**2))
    )

    mcc = (
        (c * s - np.sum(t * p)) / denominator
        if denominator else 0
    )

    # Rank-based one-vs-rest AUC
    def binary_auc(binary_y, scores):
        positives = scores[binary_y == 1]
        negatives = scores[binary_y == 0]

        if len(positives) == 0 or len(negatives) == 0:
            return np.nan

        comparisons = (
            positives[:, None] - negatives[None, :]
        )

        return (
            np.sum(comparisons > 0)
            + 0.5 * np.sum(comparisons == 0)
        ) / (len(positives) * len(negatives))

    auc_scores = []

    for i, c in enumerate(classes):
        binary_y = (y_true == c).astype(int)
        auc_scores.append(binary_auc(binary_y, y_prob[:, i]))

    return {
        "Accuracy": accuracy,
        "Macro F1": np.mean(f1_scores),
        "Macro Recall": np.mean(recalls),
        "MCC": mcc,
        "Macro AUC": np.nanmean(auc_scores)
    }

In [5]:
def stratified_folds(y, n_splits=5, seed=42):
    rng = np.random.default_rng(seed)
    folds = [[] for _ in range(n_splits)]

    for c in np.unique(y):
        indices = np.where(y == c)[0]
        rng.shuffle(indices)

        for i, index in enumerate(indices):
            folds[i % n_splits].append(index)

    return [
        np.array(sorted(fold), dtype=int)
        for fold in folds
    ]


def cross_validate_tree(
    X, y, criterion, depth,
    n_splits=5, repeats=3, seed=42
):
    results = []

    for repeat in range(repeats):
        folds = stratified_folds(
            y,
            n_splits=n_splits,
            seed=seed + repeat
        )

        for fold_id, test_idx in enumerate(folds):
            train_idx = np.setdiff1d(
                np.arange(len(y)), test_idx
            )

            tree = DecisionTreeScratch(
                criterion=criterion,
                max_depth=depth,
                min_samples_split=5,
                min_samples_leaf=3
            )

            tree.fit(X[train_idx], y[train_idx])

            pred = tree.predict(X[test_idx])
            prob = tree.predict_proba(X[test_idx])

            scores = classification_metrics(
                y[test_idx],
                pred,
                prob,
                tree.classes_
            )

            scores["Repeat"] = repeat + 1
            scores["Fold"] = fold_id + 1
            results.append(scores)

    return pd.DataFrame(results)

In [8]:
depth_values = range(1, 16)
criteria = ["gini", "entropy"]

depth_results = []

for criterion in criteria:
    for depth in depth_values:
        cv_results = cross_validate_tree(
            X, y, criterion, depth,
            n_splits=5,
            repeats=3
        )

        depth_results.append({
            "Criterion": criterion,
            "Depth": depth,
            "Mean Macro F1": cv_results["Macro F1"].mean(),
            "SD Macro F1": cv_results["Macro F1"].std()
        })

depth_df = pd.DataFrame(depth_results)

display(depth_df.round(3))

plt.figure(figsize=(9, 5))

for criterion in criteria:
    subset = depth_df[depth_df["Criterion"] == criterion]

    plt.plot(
        subset["Depth"],
        1 - subset["Mean Macro F1"],
        marker="o",
        label=criterion.capitalize()
    )

plt.xlabel("Maximum tree depth")
plt.ylabel("Macro F1 error (1 - macro F1)")
plt.title("Tree Depth Selection: Gini vs Entropy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

best_depths = {}

for criterion in criteria:
    subset = depth_df[depth_df["Criterion"] == criterion]
    best_row = subset.loc[subset["Mean Macro F1"].idxmax()]
    best_depths[criterion] = int(best_row["Depth"])

print("Selected depths:", best_depths)

KeyboardInterrupt: 

In [7]:
final_results = []

for criterion in ["gini", "entropy"]:
    scores = cross_validate_tree(
        X, y,
        criterion=criterion,
        depth=best_depths[criterion],
        n_splits=5,
        repeats=3
    )

    for metric in [
        "Accuracy", "Macro F1", "Macro Recall", "MCC", "Macro AUC"
    ]:
        final_results.append({
            "Criterion": criterion,
            "Metric": metric,
            "Mean": scores[metric].mean(),
            "SD": scores[metric].std()
        })

results_df = pd.DataFrame(final_results)

display(results_df.round(3))

pivot = results_df.pivot(
    index="Metric",
    columns="Criterion",
    values="Mean"
)

pivot.plot(kind="bar", figsize=(10, 5))
plt.ylabel("Mean cross-validation score")
plt.title("Decision Tree from Scratch: Gini vs Entropy")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

NameError: name 'best_depths' is not defined

In [ ]:
def plot_scratch_tree(tree, feature_names, title, max_display_depth=3):
    fig, ax = plt.subplots(figsize=(20, 10))
    ax.axis("off")

    def draw(node, x, y, width, depth):
        label = f"Class: {node['prediction']}\n"
        label += f"Samples: {node['samples']}\n"
        label += f"Impurity: {node['impurity']:.3f}"

        if node["feature"] is not None:
            label = (
                f"{feature_names[node['feature']]} "
                f"≤ {node['threshold']:.3f}\n"
                + label
            )

        ax.text(
            x, y, label,
            ha="center", va="center", fontsize=8,
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor="#FFF2CC" if node["feature"] is not None
                else "#D9EAD3",
                edgecolor="#333333",
                linewidth=1.2
            )
        )

        if node["feature"] is None or depth >= max_display_depth:
            return

        child_y = y - 1.5
        left_x = x - width / 2
        right_x = x + width / 2

        ax.plot([x, left_x], [y - 0.2, child_y + 0.2], color="black")
        ax.plot([x, right_x], [y - 0.2, child_y + 0.2], color="black")

        ax.text(
            (x + left_x) / 2, (y + child_y) / 2,
            "Yes", fontsize=8, ha="center"
        )
        ax.text(
            (x + right_x) / 2, (y + child_y) / 2,
            "No", fontsize=8, ha="center"
        )

        draw(node["left"], left_x, child_y, width / 2, depth + 1)
        draw(node["right"], right_x, child_y, width / 2, depth + 1)

    draw(tree.root, 0, 0, 10, 0)

    ax.set_xlim(-11, 11)
    ax.set_ylim(-max_display_depth * 1.7 - 1, 1.5)
    ax.set_title(title, fontsize=15, pad=15)

    plt.tight_layout()
    plt.show()


for criterion in ["gini", "entropy"]:
    tree = DecisionTreeScratch(
        criterion=criterion,
        max_depth=best_depths[criterion],
        min_samples_split=5,
        min_samples_leaf=3
    )

    tree.fit(X, y)

    plot_scratch_tree(
        tree,
        feature_names,
        f"Decision Tree from Scratch — {criterion.capitalize()}",
        max_display_depth=3
    )